## Load in Test Case & Run Power Flow

In [1]:
# path setup
from pathlib import Path
project_dir = Path.cwd()
test_case_dir = project_dir / "test_cases" # all test case excel sheets
out_dir = project_dir / "out"

In [2]:
# setting up ANDES
%matplotlib inline

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import andes

from calculate_metrics import calculate_metrics

andes.config_logger(stream_level=30)

In [3]:
# base case file
case_path = test_case_dir / "regf2_testbench.xlsx"

## Run TDS Sweep

In [ ]:
# anything w/o units is in p.u. 
metrics, metrics_42, metrics_1, metrics_123 = pd.DataFrame({
    "Tr (s)": [], 
    "Pref": [],
    "Peak Inertial Power": [],
    "Power Available":[],
    "H_RMSE (s)": [],
    "Max RoCoF":[],
    "Nadir":[],
    "Settling Time (s)":[]
})

# save a few time-series for visualization
M_series, M_series_42, M_series_1, M_series_123 = pd.DataFrame({
})


In [ ]:
STEP_TIME = 0.01 # 10 ms
Tr_vals = np.linspace(0, 0.1, 51)
#Tr_vals = np.linspace(0, 0.04, 11) # rerun to get more granular time-series
M_rec_vals = np.linspace(0, 0.1, 11)

print(Tr_vals)
print(M_rec_vals)

In [ ]:
# i'm too lazy to write in another layer for different random seeds
# so fyi we're running this with seeds 42, 0, and 315 (my favorite day :O)

metrics = metrics.iloc[0:0]

for tr in Tr_vals:
    # load a new case
    base_case = andes.get_case(case_path)
    ss = andes.load(base_case, setup=False)

    # change Tr
    ss.REGF2.set('Tr', 1, tr) # independent variable
    record_M = tr in M_rec_vals

    print(f"Running TDS for Tr = {ss.REGF2.Tr.v}. Recording = {record_M}")

    # TDS
    ss.setup()
    ss.config.warn_abnormal = 1
    ss.PFlow.run()

    ss.TDS.config.tf = 10 # Baruzzi et al ran simulation for 9s after disturbance
    ss.TDS.config.fixt = 1 # fixed step size
    ss.TDS.config.tstep = STEP_TIME # matches Baruzzi et al. 

    ss.TDS.config.no_tqdm = 1
    ss.TDS.config.criteria = 1
    ss.TDS.run()

    # metrics
    metrics, M_series = calculate_metrics(ss, record_M=record_M)

print("All done!")

In [ ]:
metrics.to_excel(f"{project_dir}/out/test_1/1_metrics_10_granular.xlsx")

In [ ]:
# assign run vars to these
# yes I know this is not the greatest way to do things
metrics_r2 = metrics
M_series_42 = M_series

## Visualizaton

In [ ]:
metrics.set_index("Tr (s)")
convergence_failed = 0.062 # Tr value after which TDS starts to fail to converge
x_tr = metrics["Tr (s)"]

### Peak Delivered Power vs. Tr

In [ ]:
y_1 = metrics["Peak Inertial Power"]
y_power_avail = metrics["Power Available"]

fig, ax1 = plt.subplots(figsize=(6, 3))

# First Y-Axis (Left side)
color = 'tab:blue'
ax1.set_xlabel('Tr (s)')
ax1.set_ylabel('Peak Inertial Power (p.u.)') # y-axis
ax1.plot(x_tr, y_1, color=color, label="Highest REGF2.Pe")
ax1.tick_params(axis='y')

ax1.plot(x_tr, y_power_avail, color="tab:red", label="p_max", linestyle="--") # power available 
first_power_over = np.asarray(x_tr[y_1 >= y_power_avail]).min()
ax1.axvspan(first_power_over, 0.1, color='red', alpha=0.1)

ax1.axvspan(convergence_failed, 0.1, color='gray', alpha=0.2)
ax1.legend(loc="lower right", fontsize=7, framealpha=0.7)

ax1.set_xlim(0, 0.1)

### Max RoCoF and Nadir vs. Tr

In [ ]:
y_1 = metrics["Max RoCoF"]
y_2 = metrics["Nadir"]

fig, ax1 = plt.subplots(figsize=(6, 3))

# First Y-Axis (Left side)
color = 'tab:blue'
ax1.set_xlabel('Tr (s)')
ax1.set_ylabel('Max RoCoF (p.u.)', color=color)
ax1.plot(x_tr, y_1, color=color)
ax1.tick_params(axis='y', labelcolor=color)

# Second Y-Axis (Right side)
ax2 = ax1.twinx()  # Instantiate a second axes that shares the same x-axis
color = 'tab:orange'
ax2.set_ylabel('Nadir (p.u.)', color=color)
ax2.plot(x_tr, y_2, color=color)
ax2.tick_params(axis='y', labelcolor=color)

ax2.set_ylim(0.9, 1.05)
ax2.set_xlim(0, 0.1)
ax2.axvspan(convergence_failed, 0.2, color='gray', alpha=0.2)
ax1.axvspan(first_power_over, 0.1, color='red', alpha=0.1)

ax1.set_ylim(0.0, 0.03)

fig.tight_layout()  # Prevents right-hand label from getting clipped
plt.show()

### RMSE and Settling Time vs. Tr

In [ ]:
x_tr = metrics["Tr (s)"]
y_rmse = metrics["H_RMSE (s)"]
y_settle = metrics["Settling Time (s)"]

fig, ax1 = plt.subplots(figsize=(6, 3))

# First Y-Axis (Left side)
color = 'tab:blue'
ax1.set_xlabel('Tr (s)')
ax1.set_ylabel('Inertia RMSE (s)', color=color)
ax1.plot(x_tr, y_rmse, color=color)
ax1.tick_params(axis='y', labelcolor=color)

# Second Y-Axis (Right side)
ax2 = ax1.twinx()  # Instantiate a second axes that shares the same x-axis
color = 'tab:orange'
ax2.set_ylabel('Settling Time (s)', color=color)
ax2.plot(x_tr, y_settle, color=color)
ax2.tick_params(axis='y', labelcolor=color)

ax2.set_ylim(0.9, 1)
ax2.set_xlim(0, 0.1)
ax2.axvspan(convergence_failed, 0.2, color='gray', alpha=0.2)
ax1.axvspan(first_power_over, 0.1, color='red', alpha=0.1)

fig.tight_layout()  # Prevents right-hand label from getting clipped
plt.show()

### Plotting Estimated H Time-Series for Selected Values of Tr 

In [ ]:
H_gfm = 5
time_invalid = 1 + STEP_TIME * 20

cmap = plt.cm.RdYlBu

fig, ax = plt.subplots(figsize=(6, 3))

for i, tr_val in enumerate(h_series.columns):
    if i % 3 == 0:
        color = cmap(i / (len(h_series.columns) - 1))

        a = 1 - i*0.05
        print(a)
        ax.plot(h_series.index, h_series[tr_val], color=color, linewidth=1.8,
                label=f"Tr = {tr_val:.4f}s", alpha = a, zorder = 10 - i)

plt.axline((0, H_gfm), slope=0, color="purple", linestyle=":", label=f"$H_{{nominal}}$")
ax.axvspan(0, time_invalid, color='gray', alpha=0.2)
    
ax.set_xlabel("Time (s)")
ax.set_ylabel("Estimated Inertia (s)")

ax.set_title(f"Estimated Inertia For Values of Tr")
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)
ax.set_xlim(0, 10)
ax.set_ylim(-10, 20)